<img src="https://www.plus2net.com/images/top2.jpg" alt="plus2net logo" width="400">

# Python SQLite Updating Records & Pandas Bulk Upsert Practice

**Main Tutorial Link:** [Python SQLite Updating Records](https://www.plus2net.com/python/sqlite-update.php)

---

## 1. Setup Database & Download `my_db.db`

In [ ]:
!wget https://www.plus2net.com/python/download/my_db.db

## 2. Update Single Column (`sqlite3`)

In [ ]:
import sqlite3

my_conn = sqlite3.connect('my_db.db')
q = "UPDATE student SET mark = mark + 5 WHERE id = 7"
r_set = my_conn.execute(q)
my_conn.commit()
print("Records updated:", r_set.rowcount)
my_conn.close()

## 3. Updating Multiple Columns with Error Handling

In [ ]:
import sqlite3

my_conn = sqlite3.connect('my_db.db')
q = "UPDATE student SET class = 'Five' WHERE class = 'Four'"
try:
    r_set = my_conn.execute(q)
    my_conn.commit()
    print("Records updated:", r_set.rowcount)
except sqlite3.Error as my_error:
    print("error: ", my_error)
finally:
    my_conn.close()

## 4. Modern SQLAlchemy 2.0+ Context Manager Update

In [ ]:
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError

my_conn = create_engine("sqlite:///my_db.db")

try:
    with my_conn.begin() as connection:
        q = text("UPDATE student SET mark = mark + 1")
        r_set = connection.execute(q)
        print("No of records updated : ", r_set.rowcount)
except SQLAlchemyError as e:
    print("SQLAlchemy Error:", e)

## 5. Using Parameterized Queries to Prevent SQL Injection

In [ ]:
import sqlite3

my_conn = sqlite3.connect('my_db.db')
my_data = ('Seven', 'Six') # tuple to pass values to execute method
q = "UPDATE student SET class = ? WHERE class = ?"
try:
    r_set = my_conn.execute(q, my_data)
    my_conn.commit()
    print("Records updated : ", r_set.rowcount)
except sqlite3.Error as my_error:
    print("error: ", my_error)
finally:
    my_conn.close()

## 6. Bulk Updates and Upserts using Pandas DataFrame (`to_sql`)

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

my_conn = create_engine("sqlite:///my_db.db")

# Create sample update DataFrame
df_update = pd.DataFrame({
    'id': [1, 2, 3],
    'name': ['John Deo', 'Max Ruin', 'Arnold'],
    'class': ['Five', 'Six', 'Four'],
    'mark': [80, 90, 65],
    'gender': ['female', 'male', 'male']
})

# Bulk insert/upsert into temporary staging table and run UPDATE ON CONFLICT
with my_conn.begin() as conn:
    df_update.to_sql('temp_student', conn, if_exists='replace', index=False)
    upsert_sql = text("""
        INSERT INTO student (id, name, class, mark, gender)
        SELECT id, name, class, mark, gender FROM temp_student
        WHERE true
        ON CONFLICT(id) DO UPDATE SET
            name = excluded.name,
            class = excluded.class,
            mark = excluded.mark,
            gender = excluded.gender;
    """
    conn.execute(upsert_sql)
    conn.execute(text("DROP TABLE temp_student"))
print("Pandas bulk upsert complete!")

---
### Visit [www.plus2net.com](https://www.plus2net.com) for more Python tutorials, code snippets, and data science guides.